# 1장 2강: 활성화 함수와 순전파 — 실습문제 풀이

> **풀이 기준**: 프로젝트에 업로드된 `concrete_data.csv`로 코드를 실제 실행한 결과입니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 문제에 주어진 W1·b1·W2·b2를 그대로 사용했습니다.

---

## 실습 준비

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()

print("전체 데이터 크기:", df.shape)
print(df[["cement", "water", "concrete_compressive_strength"]].head(3).to_string(index=False))

전체 데이터 크기: (1030, 9)
 cement  water  concrete_compressive_strength
  540.0  162.0                          79.99
  540.0  162.0                          61.89
  332.5  228.0                          40.27


---

## 실습 1. 활성화 함수와 비선형의 의미 이해하기

### 문제 1-1. Sigmoid·Tanh·ReLU 출력 비교

In [2]:
z = np.array([-2, -1, 0, 1, 2], dtype=float)

sigmoid_values = 1 / (1 + np.exp(-z))
tanh_values = np.tanh(z)
relu_values = np.maximum(0, z)

print("z       :", z)
print("Sigmoid :", np.round(sigmoid_values, 4))
print("Tanh    :", np.round(tanh_values, 4))
print("ReLU    :", np.round(relu_values, 4))

z       : [-2. -1.  0.  1.  2.]
Sigmoid : [0.1192 0.2689 0.5    0.7311 0.8808]
Tanh    : [-0.964  -0.7616  0.      0.7616  0.964 ]
ReLU    : [0. 0. 0. 1. 2.]


#### 답변

**수식 및 계산 과정**

- z = 0일 때
  - Sigmoid: 1 / (1 + e⁰) = 1 / (1 + 1) = **0.5**
  - Tanh: (e⁰ − e⁰) / (e⁰ + e⁰) = (1 − 1) / (1 + 1) = **0**
  - ReLU: max(0, 0) = **0**
- ReLU 원소별 계산: max(0,−2)=0, max(0,−1)=0, max(0,0)=0, max(0,1)=1, max(0,2)=2 → **[0, 0, 0, 1, 2]**

**Q1. z=0일 때 세 함수의 결과는 각각 무엇인가요?**

- Sigmoid 0.5, Tanh 0, ReLU 0입니다. e⁰ = 1을 대입하면 위 계산과 같습니다.

**Q2. 음수를 입력했을 때 세 함수의 결과는 어떻게 다른가요? z=−2를 예로 설명하세요.**

- Sigmoid: 0.1192로, 0보다 크지만 0에 가까운 양수입니다(항상 0~1 사이).
- Tanh: −0.964로, 음수 부호가 유지되며 −1에 가까워집니다(−1~1 사이).
- ReLU: 0으로, 음수 입력은 모두 0이 됩니다.

**Q3. ReLU의 결과가 `[0, 0, 0, 1, 2]`인 이유는 무엇인가요? 양수 2도 1로 줄이나요?**

- ReLU는 0과 z 중 큰 값을 고르므로 음수와 0은 0이 되고, 양수는 그대로 통과합니다. 2는 줄지 않고 **그대로 2**입니다. ReLU는 위쪽 상한이 없습니다.

**Q4. Sigmoid의 값이 0과 1 사이이면 그 자체로 신뢰할 수 있는 구매 확률이나 압축강도인가요?**

- 아닙니다. Sigmoid는 어떤 숫자든 0~1 범위로 바꿀 뿐입니다. 그 값이 의미 있는 확률이 되려면 해당 문제에 맞게 학습된 모델이어야 합니다. 이번 z는 함수 관찰용 예시 숫자이고, 압축강도는 0~1 범위의 값도 아닙니다.

---

### 문제 1-2. 비선형 활성화 함수가 필요한 이유 확인

In [3]:
x_demo = np.array([-2, -1, 0, 1, 2], dtype=float)

z_demo = 3 * x_demo + 1
out_plain = 2 * z_demo + 4
out_merged = 6 * x_demo + 6

h_demo = np.maximum(0, z_demo)
out_relu = 2 * h_demo + 4

print("x_demo     :", x_demo)
print("z_demo     :", z_demo)
print("h_demo     :", h_demo)
print("out_plain  :", out_plain)
print("out_merged :", out_merged)
print("out_relu   :", out_relu)
print("out_plain과 out_merged가 같은가?:", np.allclose(out_plain, out_merged))

x_demo     : [-2. -1.  0.  1.  2.]
z_demo     : [-5. -2.  1.  4.  7.]
h_demo     : [0. 0. 1. 4. 7.]
out_plain  : [-6.  0.  6. 12. 18.]
out_merged : [-6.  0.  6. 12. 18.]
out_relu   : [ 4.  4.  6. 12. 18.]
out_plain과 out_merged가 같은가?: True


#### 답변

**수식 및 계산 과정 (x = −2)**

- 활성화 없음: z₁ = 3(−2) + 1 = −5 → h₁ = −5 → out = 2(−5) + 4 = **−6**
- ReLU 사용: z₁ = −5 → h₁ = max(0, −5) = 0 → out = 2(0) + 4 = **4**

**Q1. 활성화가 없을 때 두 단계의 식은 어떤 한 식으로 정리되나요? 배열 비교 결과는 무엇인가요?**

- 2(3x + 1) + 4 = 6x + 6으로 정리됩니다. `np.allclose(out_plain, out_merged)`는 **True**입니다. 두 단계를 거쳐도 결국 한 번의 직선 계산과 같습니다.

**Q2. x=−2일 때 활성화가 없는 출력과 ReLU를 넣은 출력의 계산 과정을 각각 적으세요.**

- 위의 계산 과정과 같습니다. 활성화 없음 −6, ReLU 사용 4입니다.

**Q3. 다섯 입력을 차례로 볼 때 out_plain과 out_relu는 어떻게 증가하나요? ReLU를 넣으면 왜 하나의 직선으로 정리할 수 없나요?**

- out_plain: −6 → 0 → 6 → 12 → 18로, 입력이 1 늘 때마다 **항상 6씩** 증가합니다.
- out_relu: 4 → 4 → 6 → 12 → 18로, 증가량이 **0, 2, 6, 6**으로 구간마다 다릅니다.
- 증가량이 구간마다 다르므로(z₁ < 0인 구간은 평평하고, z₁ > 0인 구간은 기울기 6) 전체를 하나의 직선 ax + b로 표현할 수 없습니다.

**Q4. 층만 여러 개 쌓고 비선형 활성화 함수를 전혀 쓰지 않으면 어떤 한계가 있나요? 비선형 함수를 넣으면 항상 예측 성능이 좋아지나요?**

- 층을 아무리 쌓아도 전체가 하나의 선형(아핀) 계산으로 합쳐지므로, 한 층짜리 모델이 표현할 수 있는 관계 이상을 표현하지 못합니다. 곡선이나 꺾이는 관계를 나타낼 수 없습니다.
- 비선형 함수는 표현할 수 있는 관계의 범위를 넓혀 줄 뿐, 성능 향상을 보장하지는 않습니다. 성능은 학습, 데이터, 구조 등에 따라 달라집니다.

---

## 실습 2. 콘크리트 데이터로 신경망 순전파 계산하기

### 문제 2-1. 첫 번째 샘플의 입력부터 최종 출력까지 계산

In [4]:
X_one = df[["cement", "water"]].to_numpy()[0].reshape(1, 2)
y_true = df["concrete_compressive_strength"].to_numpy()[0]

W1 = np.array([[0.01, -0.01],
               [0.00,  0.02]])
b1 = np.array([-3.0, 1.0])
W2 = np.array([[2.0],
               [1.0]])
b2 = np.array([0.5])

Z1 = X_one @ W1 + b1
H1 = np.maximum(0, Z1)
Z2 = H1 @ W2 + b2
y_hat = Z2
y_no_activation = Z1 @ W2 + b2

print("X_one:", X_one, " shape:", X_one.shape)
print("y_true:", y_true)
print("Z1:", np.round(Z1, 4), " shape:", Z1.shape)
print("H1:", np.round(H1, 4), " shape:", H1.shape)
print("y_hat:", np.round(y_hat, 4), " shape:", y_hat.shape)
print("y_no_activation:", np.round(y_no_activation, 4))

X_one: [[540. 162.]]  shape: (1, 2)
y_true: 79.99
Z1: [[ 2.4  -1.16]]  shape: (1, 2)
H1: [[2.4 0. ]]  shape: (1, 2)
y_hat: [[5.3]]  shape: (1, 1)
y_no_activation: [[4.14]]


#### 답변

**수식 및 계산 과정**

- 은닉 뉴런 1: 540 × 0.01 + 162 × 0.00 + (−3) = 5.4 + 0 − 3 = **2.4**
- 은닉 뉴런 2: 540 × (−0.01) + 162 × 0.02 + 1 = −5.4 + 3.24 + 1 = **−1.16**
- ReLU: H1 = [max(0, 2.4), max(0, −1.16)] = **[2.4, 0]**
- 출력: 2.4 × 2.0 + 0 × 1.0 + 0.5 = 4.8 + 0 + 0.5 = **5.3**
- 활성화 생략: 2.4 × 2.0 + (−1.16) × 1.0 + 0.5 = 4.8 − 1.16 + 0.5 = **4.14**

**Q1. 첫 번째 은닉 뉴런과 두 번째 은닉 뉴런의 활성화 전 값은 각각 얼마인가요? W1의 어느 값끼리 사용하나요?**

- 은닉 뉴런 1은 W1의 **첫 번째 열** [0.01, 0.00]과 편향 −3을 사용해 **2.4**입니다.
- 은닉 뉴런 2는 W1의 **두 번째 열** [−0.01, 0.02]와 편향 1을 사용해 **−1.16**입니다.
- W1의 행은 입력(시멘트, 물), 열은 은닉 뉴런에 대응합니다.

**Q2. ReLU 이후 어떤 값이 달라지며, H1의 Shape는 무엇인가요?**

- 음수였던 은닉 뉴런 2의 값 −1.16이 **0으로 바뀝니다**. 뉴런 1의 2.4는 그대로입니다.
- H1의 Shape는 **(1, 2)**로, ReLU는 원소별로 적용되어 크기가 유지됩니다.

**Q3. 최종 출력과 은닉 활성화를 생략한 출력은 각각 얼마인가요? 달라진 이유는 무엇인가요?**

- 최종 출력 **5.3**, 활성화 생략 출력 **4.14**입니다.
- ReLU가 은닉 뉴런 2의 음수 값(−1.16)을 0으로 바꿨기 때문입니다. 생략하면 −1.16 × 1.0이 그대로 더해져 1.16만큼 작아집니다.

**Q4. 출력층에 Sigmoid를 적용하면 출력 범위는 어떻게 되나요? 이번에 출력층 활성화를 따로 적용하지 않은 이유는 무엇인가요?**

- 출력이 **0과 1 사이**로 제한됩니다(예: sigmoid(5.3) ≈ 0.995).
- 압축강도는 0~1로 제한되지 않는 연속적인 수치이므로, 회귀 문제에서는 보통 출력층 계산값을 그대로 사용합니다.

**Q5. 실제 정답 79.99는 순전파의 어느 계산에 들어가나요? 순전파를 실행한 것이 학습을 수행한 것인가요?**

- 순전파의 **어느 계산에도 들어가지 않습니다**. 출력과 비교하기 위해 확인만 했습니다.
- 순전파는 고정된 가중치로 출력을 계산하는 것일 뿐이며, 가중치를 바꾸지 않았으므로 학습이 아닙니다.

---

## 과제. 같은 순전파를 다른 샘플에 적용하기

### 문제 3-1. 세 번째 샘플의 순전파와 활성화 전후 비교

In [5]:
X_task = df[["cement", "water"]].to_numpy()[2].reshape(1, 2)
y_task_true = df["concrete_compressive_strength"].to_numpy()[2]

W1 = np.array([[0.01, -0.01],
               [0.00,  0.02]])
b1 = np.array([-3.0, 1.0])
W2 = np.array([[2.0],
               [1.0]])
b2 = np.array([0.5])

Z1_task = X_task @ W1 + b1
H1_task = np.maximum(0, Z1_task)
y_task_hat = H1_task @ W2 + b2
y_task_no_activation = Z1_task @ W2 + b2

print("X_task:", X_task, " shape:", X_task.shape)
print("y_task_true:", y_task_true)
print("Z1_task:", np.round(Z1_task, 4), " shape:", Z1_task.shape)
print("H1_task:", np.round(H1_task, 4), " shape:", H1_task.shape)
print("y_task_hat:", np.round(y_task_hat, 4), " shape:", y_task_hat.shape)
print("y_task_no_activation:", np.round(y_task_no_activation, 4))
print("두 출력이 같은가?:", np.allclose(y_task_hat, y_task_no_activation))

X_task: [[332.5 228. ]]  shape: (1, 2)
y_task_true: 40.27
Z1_task: [[0.325 2.235]]  shape: (1, 2)
H1_task: [[0.325 2.235]]  shape: (1, 2)
y_task_hat: [[3.385]]  shape: (1, 1)
y_task_no_activation: [[3.385]]
두 출력이 같은가?: True


#### 답변

**수식 및 계산 과정**

- 은닉 뉴런 1: 332.5 × 0.01 + 228 × 0.00 = 3.325 → 편향 더하기: 3.325 − 3 = **0.325**
- 은닉 뉴런 2: 332.5 × (−0.01) + 228 × 0.02 = −3.325 + 4.56 = 1.235 → 편향 더하기: 1.235 + 1 = **2.235**
- ReLU: 두 값 모두 양수이므로 H1 = **[0.325, 2.235]**
- 출력: 0.325 × 2 + 2.235 × 1 + 0.5 = 0.65 + 2.235 + 0.5 = **3.385**
- 활성화 생략: Z1 = H1이므로 같은 **3.385**

**Q1. 두 은닉 뉴런의 활성화 전 값은 각각 얼마인가요?**

- 뉴런 1: 곱셈 합 3.325, 편향 더한 뒤 **0.325**
- 뉴런 2: 곱셈 합 1.235, 편향 더한 뒤 **2.235**

**Q2. ReLU를 적용한 뒤 두 값은 바뀌나요? 이유는 무엇인가요?**

- 바뀌지 않습니다. 두 값이 모두 양수이므로 ReLU가 그대로 통과시킵니다.

**Q3. 최종 출력은 얼마이며 은닉 활성화를 생략한 출력과 같은가요?**

- 최종 출력은 **3.385**이며, 활성화를 생략한 출력도 3.385로 같습니다. `np.allclose()` 결과는 **True**입니다.

**Q4. 이 샘플에서 두 출력이 같으면 모든 입력에서 ReLU가 필요 없다고 결론 내릴 수 있나요?**

- 아닙니다. 첫 샘플에서는 은닉 뉴런 2가 −1.16이었기 때문에 ReLU 유무에 따라 출력이 5.3과 4.14로 달라졌습니다. ReLU의 효과는 **음수가 나오는 입력에서만** 드러나므로, 한 샘플의 결과로 일반화할 수 없습니다.

**Q5. 실제 정답 40.27과 최종 출력은 역할이 어떻게 다른가요? 샘플이 바뀌면서 바꾼 값과 유지한 값은 무엇인가요?**

- 40.27은 데이터에 기록된 측정값이고, 3.385는 학습하지 않은 고정 파라미터로 계산한 모델의 출력입니다. 정답은 계산에 쓰이지 않았습니다.
- 바꾼 것은 입력 샘플(332.5, 228)이고, 구조·W1·b1·W2·b2·ReLU는 모두 유지했습니다.

---

## 실습 마무리

**1. Sigmoid·Tanh·ReLU의 출력 범위를 각각 설명하세요.**

- Sigmoid: 0보다 크고 1보다 작음 / Tanh: −1보다 크고 1보다 작음 / ReLU: 0 이상, 위쪽 제한 없음

**2. 비선형 활성화 함수 없이 가중합·편향 층만 쌓으면 어떤 일이 생기나요?**

- 여러 층이 하나의 직선(아핀) 식으로 합쳐집니다. 실습 1-2에서 2(3x + 1) + 4 = 6x + 6이 된 것처럼, 층을 쌓은 효과가 사라집니다.

**3. Z1과 H1은 무엇이 다른가요?**

- Z1은 가중합에 편향을 더한 **활성화 전** 값이고, H1은 Z1에 ReLU를 적용한 **활성화 후** 값입니다. 다음 층에는 H1이 전달됩니다.

**4. 순전파의 계산 순서를 적으세요.**

- 입력 X → Z1 = X @ W1 + b1 → H1 = ReLU(Z1) → Z2 = H1 @ W2 + b2 → ŷ = Z2

**5. ReLU 이후 값이 그대로라면 코드 오류인가요?**

- 아닙니다. 입력이 모두 양수이면 ReLU는 값을 그대로 통과시킵니다. 과제의 세 번째 샘플이 그 예입니다.

**6. 순전파와 학습은 같은 뜻인가요?**

- 다릅니다. 순전파는 주어진 가중치로 출력을 계산하는 과정이고, 학습은 출력과 정답의 차이를 이용해 가중치를 조정하는 과정입니다. 이번 실습에서는 순전파만 수행했습니다.